# RP-010 — Google Colab GPU preflight

Before running this notebook, select **Runtime → Change runtime type → GPU**. This notebook does not load a model, start paid resources, expose a public tunnel, or automatically switch models. It checks whether the selected runtime is eligible for an RP-011 model-load test.

In [ ]:
from pathlib import Path
import os
import subprocess

REPOSITORY_URL = 'REPLACE_WITH_YOUR_REPOSITORY_URL'
REPOSITORY_ROOT = Path('/content/ATS-CV')
PROJECT_ROOT = REPOSITORY_ROOT / 'PDFPARSER'

if not (PROJECT_ROOT / 'requirements-ml.txt').is_file():
    if REPOSITORY_URL.startswith('REPLACE_'):
        raise RuntimeError('Set REPOSITORY_URL before cloning the project')
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPOSITORY_ROOT)], check=True)

os.chdir(PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
%pip install -q -r requirements-dev.txt -r requirements-ml.txt

In [ ]:
# Keep checkpoint downloads in the ephemeral Colab runtime by default.
# Change this only when you have explicitly authorized persistent storage.
HF_CACHE_ROOT = Path('/content/huggingface')
HF_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_CACHE_ROOT)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE_ROOT / 'hub')
os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE_ROOT / 'transformers')
print(f'Hugging Face cache: {HF_CACHE_ROOT}')

In [ ]:
# Optional: use a Colab Secret named HF_TOKEN only for gated/private models.
# The token is never printed. Qwen2.5 public checkpoints normally do not require it.
USE_HF_TOKEN = False
if USE_HF_TOKEN:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if not token:
        raise RuntimeError('HF_TOKEN was requested but is not configured in Colab Secrets')
    os.environ['HF_TOKEN'] = token

In [ ]:
!nvidia-smi

In [ ]:
import json
from app.llm.runtime_preflight import run_preflight

# Qwen14B is always evaluated first. Change this to 'qwen7b' yourself only
# after recording a blocked Qwen14B report. There is no automatic fallback.
MODEL_ALIAS = 'qwen14b'
REPORT_PATH = PROJECT_ROOT / 'artifacts' / 'rp010_preflight.json'
report = run_preflight(MODEL_ALIAS, output_path=REPORT_PATH)
print(json.dumps(report, indent=2, ensure_ascii=False))
print(f'Report saved to: {REPORT_PATH}')

In [ ]:
if report['decision'] != 'supported_for_load_test':
    raise RuntimeError(
        'RP-010 blocked: ' + '; '.join(report['reasons']) +
        '. Do not start real-model mode on this runtime.'
    )
print('RP-010 passed: this runtime may proceed to the RP-011 model-load test.')

In [ ]:
# Optional persistence. Enable only for an authorized Google Drive location.
MOUNT_AUTHORIZED_DRIVE = False
if MOUNT_AUTHORIZED_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount('/content/drive')
    destination = Path('/content/drive/MyDrive/ATS-CV/research-artifacts/rp010_preflight.json')
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(REPORT_PATH, destination)
    print(f'Report copied to: {destination}')

In [ ]:
# Prepare configuration for RP-011. This does not load a model yet.
os.environ['INFERENCE_MODE'] = 'transformers'
os.environ['MODEL_ALIAS'] = MODEL_ALIAS
os.environ['MODEL_REVISION'] = 'main'
os.environ['QUANTIZATION'] = 'nf4-4bit'
print({
    'INFERENCE_MODE': os.environ['INFERENCE_MODE'],
    'MODEL_ALIAS': os.environ['MODEL_ALIAS'],
    'MODEL_REVISION': os.environ['MODEL_REVISION'],
    'QUANTIZATION': os.environ['QUANTIZATION'],
})

## Stop here

RP-010 ends after recording the explicit preflight decision. Model loading belongs to RP-011 and must not be improvised in this notebook.